In [331]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moonknightmarvel/paysim/paysim.csv


In [332]:
import os
file_path = "/kaggle/input/datasets/moonknightmarvel/paysim"
print(os.listdir(file_path))

['paysim.csv']


In [333]:
import pandas as pd

file_path = "/kaggle/input/datasets/moonknightmarvel/paysim/paysim.csv"

df = pd.read_csv(file_path)

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [334]:
print(df.shape)

(6362620, 11)


In [335]:
print(df.columns)

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')


In [336]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [337]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [338]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [339]:
df.groupby('type')['amount'].mean()

type
CASH_IN     168920.242004
CASH_OUT    176273.964346
DEBIT         5483.665314
PAYMENT      13057.604660
TRANSFER    910647.009645
Name: amount, dtype: float64

In [340]:
df.groupby('type')['amount'].median()

type
CASH_IN     143427.710
CASH_OUT    147072.185
DEBIT         3048.990
PAYMENT       9482.190
TRANSFER    486308.390
Name: amount, dtype: float64

In [341]:
df.groupby('type')['isFraud'].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

In [342]:
model_df = df.copy()

In [343]:
model_df.shape

(6362620, 11)

In [344]:
model_df['orig_balance_change'] = (model_df['oldbalanceOrg'] - model_df['newbalanceOrig'])

In [345]:
model_df['dest_balance_change'] = (model_df['newbalanceDest'] - model_df['oldbalanceDest'])

In [346]:
model_df[
[
    'amount',
     'oldbalanceOrg',
     'newbalanceOrig',
     'orig_balance_change',
     'oldbalanceDest',
     'newbalanceDest',
     'dest_balance_change'
]
].head(10)

,amount,oldbalanceOrg,newbalanceOrig,orig_balance_change,oldbalanceDest,newbalanceDest,dest_balance_change
0,9839.64,170136.00,160296.36,9839.64,0.0,0.00,0.00
1,1864.28,21249.00,19384.72,1864.28,0.0,0.00,0.00
2,181.00,181.00,0.00,181.00,0.0,0.00,0.00
3,181.00,181.00,0.00,181.00,21182.0,0.00,-21182.00
4,11668.14,41554.00,29885.86,11668.14,0.0,0.00,0.00
5,7817.71,53860.00,46042.29,7817.71,0.0,0.00,0.00
6,7107.77,183195.00,176087.23,7107.77,0.0,0.00,0.00
7,7861.64,176087.23,168225.59,7861.64,0.0,0.00,0.00
8,4024.36,2671.00,0.00,2671.00,0.0,0.00,0.00
9,5337.77,41720.00,36382.23,5337.77,41898.0,40348.79,-1549.21


In [347]:
model_df['orig_balance_discrepancy'] = abs(model_df['amount'] - model_df['orig_balance_change'])

In [348]:
model_df['dest_balance_discrepancy'] = abs(model_df['amount'] - model_df['dest_balance_change'])

In [349]:
model_df[
[
    'amount',
    'orig_balance_change',
     'orig_balance_discrepancy',
     'dest_balance_change',
     'dest_balance_discrepancy'
]
].head(10)

,amount,orig_balance_change,orig_balance_discrepancy,dest_balance_change,dest_balance_discrepancy
0,9839.64,9839.64,1.455192e-11,0.00,9839.64
1,1864.28,1864.28,1.136868e-12,0.00,1864.28
2,181.00,181.00,0.000000e+00,0.00,181.00
3,181.00,181.00,0.000000e+00,-21182.00,21363.00
4,11668.14,11668.14,0.000000e+00,0.00,11668.14
5,7817.71,7817.71,9.094947e-13,0.00,7817.71
6,7107.77,7107.77,1.091394e-11,0.00,7107.77
7,7861.64,7861.64,1.364242e-11,0.00,7861.64
8,4024.36,2671.00,1.353360e+03,0.00,4024.36
9,5337.77,5337.77,3.637979e-12,-1549.21,6886.98


### Temporal Frequency

In [350]:
model_df['transactions_per_step'] = (
    model_df.groupby(['nameOrig', 'step'])['amount']
            .transform('count')
)

The above code groups rows according to both the originating account AND the hour. 'transactions_per_step' therefore means the number of transactions made by a certain user in 1 hour. See how the output looks like below

In [351]:
model_df[
    ['nameOrig', 'step', 'amount', 'transactions_per_step']
].sort_values(
    by='transactions_per_step',
    ascending=False
).head(20)

,nameOrig,step,amount,transactions_per_step
1451734,C722055608,140,73597.75,2
1389388,C389626503,139,139133.13,2
4953465,C1194642233,351,10717.29,2
1422651,C1828219334,139,12788.63,2
371827,C884511024,17,288371.66,2
2459064,C1965131583,203,5961.67,2
4416739,C2084454803,322,83824.57,2
458746,C570148238,19,212959.86,2
5834340,C1061931265,402,31002.99,2
1714270,C407378873,160,8503.44,2


In [352]:
model_df['transactions_per_step'].value_counts().sort_index()

transactions_per_step
1    6362526
2         94
Name: count, dtype: int64

The above output means: 6,362,526 rows have transactions_per_step = 1, while only 94 rows have transactions_per_step = 2.

I was trying to use frequency of transactions to detect temporal frequency cz I thought that an unsual number of transactions by the same account within an hour can represent transaction-frequency anomalies but then as you have seen, there are no unusual frequency. The maximum is 2 so we'll have to use sth else.

In [353]:
model_df = model_df.sort_values('step').copy()

model_df['time_since_last_transaction'] = (
    model_df.groupby('nameOrig')['step'].diff()
)

In [354]:
model_df['time_since_last_transaction'].describe()

count    9313.000000
mean      158.312574
std       123.660909
min         0.000000
25%        56.000000
50%       139.000000
75%       225.000000
max       704.000000
Name: time_since_last_transaction, dtype: float64

In [355]:
model_df['time_since_last_transaction'].value_counts().sort_index().head(20)

time_since_last_transaction
0.0     47
1.0     65
2.0     68
3.0     57
4.0     77
5.0     60
6.0     47
7.0     39
8.0     28
9.0     37
10.0    23
11.0    12
12.0    17
13.0    21
14.0    33
15.0    27
16.0    32
17.0    42
18.0    55
19.0    47
Name: count, dtype: int64

The above result tells us something pretty important: time_since_last_transaction is also too sparse to be a strong feature for this PaySim dataset. So only about 0.15% of transactions have a usable time gap. The other ~99.85% are the first/only transaction from that nameOrig, so their value is NaN.

For those 9,313 repeated-account transactions, the median gap is 139 hours, the mean is about 158 hours, and gaps range from 0 to 704 hours. So the calculation works, but it doesn't give us enough coverage across the dataset. Therefore, we need another way to capture temporal frequency

****Another solution:
We can move from account-level frequency to system-level transaction frequency.

Instead of asking:
How frequently is this particular account transacting?

we can ask:
How much transaction activity is occurring during this hour?

In [356]:
transactions_per_hour = model_df.groupby('step').size()

transactions_per_hour.describe()

count      743.000000
mean      8563.418573
std      13388.693217
min          2.000000
25%         12.000000
50%        529.000000
75%      12317.000000
max      51352.000000
dtype: float64

Above we have grouped the transactions per hour and then used size() to count the number of rows per hour so in short we are checking the number transactions per hour

decribe() lets us inspect how hourly transaction volume varies across the whole simulation

In [357]:
transactions_per_hour.head(10)

step
1      2708
2      1014
3       552
4       565
5       665
6      1660
7      6837
8     21097
9     37628
10    35991
dtype: int64

The difference between this one and transaction per step that we started with above is that with transaction per step, we were looking at the number of transaction made by a particular person per hour. However with this one we are checking the number of transactions that happen per hour. We haven't also grouped using the user account.

In [358]:
model_df['hourly_transaction_volume'] = (
    model_df['step'].map(transactions_per_hour)
)

Above, 'hourly_transaction_volume' holds the transaction_per_hour value. The code maps every step(hour) to it's transaction/hour value and that value is held by 'hourly_transaction_volume' column

In [359]:
model_df[
    ['step', 'amount', 'hourly_transaction_volume']
].head(10)

,step,amount,hourly_transaction_volume
23,1,3876.41,2708
22,1,1065.41,2708
21,1,9302.79,2708
20,1,1373.43,2708
19,1,215310.30,2708
18,1,671.64,2708
17,1,1157.86,2708
16,1,1563.82,2708
15,1,229133.94,2708
14,1,4098.78,2708


In [360]:
hourly_volume_change = transactions_per_hour.diff()

hourly_volume_change.head(10)

step
1         NaN
2     -1694.0
3      -462.0
4        13.0
5       100.0
6       995.0
7      5177.0
8     14260.0
9     16531.0
10    -1637.0
dtype: float64

In [361]:
hourly_volume_change_abs = hourly_volume_change.abs()

model_df['hourly_volume_change'] = (
    model_df['step'].map(hourly_volume_change_abs)
)

In [362]:
model_df[
    ['step',
     'amount',
     'hourly_transaction_volume',
     'hourly_volume_change']
].drop_duplicates(subset='step').head(10)

,step,amount,hourly_transaction_volume,hourly_volume_change
23,1,3876.41,2708,NaN
2714,2,5427.59,1014,1694.0
3741,3,194242.46,552,462.0
4281,4,1036.33,565,13.0
4972,5,357885.34,665,100.0
5516,6,12848.12,1660,995.0
7296,7,41664.88,6837,5177.0
14121,8,229393.25,21097,14260.0
60269,9,58293.61,37628,16531.0
96756,10,7125.54,35991,1637.0


#### Transaction amount variance

In [363]:
amount_stats = model_df.groupby('type')['amount'].agg(
    ['mean', 'std', 'var']
)

amount_stats

,mean,std,var
type,,,
CASH_IN,168920.242004,1.265083e+05,1.600434e+10
CASH_OUT,176273.964346,1.753297e+05,3.074052e+10
DEBIT,5483.665314,1.331854e+04,1.773834e+08
PAYMENT,13057.604660,1.255645e+04,1.576644e+08
TRANSFER,910647.009645,1.879574e+06,3.532797e+12


In [364]:
type_mean_amount = model_df.groupby('type')['amount'].transform('mean')

The above code maps the mean amounts we've gotten to their respective rows. So if a row is a cash_in transaction, type_mean_amount will hold 168920 which is the mean of cash_in transactions

In [365]:
model_df['amount_deviation'] = abs(
    model_df['amount'] - type_mean_amount
)

Basically amount_deviation means: How far is my amount from the average amount(mean) of my transaction type?

In [366]:
model_df[
    ['type', 'amount', 'amount_deviation']
].sort_values(
    'amount_deviation',
    ascending=False
).head(10)

,type,amount,amount_deviation
3686583,TRANSFER,92445516.64,9.153487e+07
4060598,TRANSFER,73823490.36,7.291284e+07
4146397,TRANSFER,71172480.42,7.026183e+07
3946920,TRANSFER,69886731.30,6.897608e+07
3911956,TRANSFER,69337316.27,6.842667e+07
3937152,TRANSFER,67500761.29,6.659011e+07
4105338,TRANSFER,66761272.21,6.585063e+07
3892529,TRANSFER,64234448.19,6.332380e+07
3991638,TRANSFER,63847992.58,6.293735e+07
4143801,TRANSFER,63294839.63,6.238419e+07


In [367]:
type_mean = model_df.groupby('type')['amount'].transform('mean')
type_std = model_df.groupby('type')['amount'].transform('std')

model_df['amount_zscore'] = (
    (model_df['amount'] - type_mean) / type_std
)

In [368]:
model_df['amount_zscore_abs'] = model_df['amount_zscore'].abs()

In [369]:
model_df[
    ['type',
     'amount',
     'amount_deviation',
     'amount_zscore_abs']
].sort_values(
    'amount_zscore_abs',
    ascending=False
).head(10)

,type,amount,amount_deviation,amount_zscore_abs
6362579,CASH_OUT,10000000.0,9.823726e+06,56.030003
6362507,CASH_OUT,10000000.0,9.823726e+06,56.030003
5987864,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266410,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266412,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266408,CASH_OUT,10000000.0,9.823726e+06,56.030003
6266406,CASH_OUT,10000000.0,9.823726e+06,56.030003
4154973,CASH_OUT,10000000.0,9.823726e+06,56.030003
6296669,CASH_OUT,10000000.0,9.823726e+06,56.030003
6296011,CASH_OUT,10000000.0,9.823726e+06,56.030003


In [370]:
model_df['amount_zscore_abs'].describe()

count    6.362620e+06
mean     6.676703e-01
std      7.444566e-01
min      1.328203e-07
25%      2.732351e-01
50%      5.434899e-01
75%      8.749816e-01
max      5.603000e+01
Name: amount_zscore_abs, dtype: float64

In [371]:
model_df.groupby('type')[
    ['dest_balance_change', 'dest_balance_discrepancy']
].agg(['mean', 'median'])

dest_balance_change             dest_balance_discrepancy  \
                        mean      median                     mean   
type                                                                
CASH_IN       -120813.411259 -104596.800            312986.774680   
CASH_OUT       193568.180775  150187.380             30283.193936   
DEBIT           19867.696753    3059.405             23945.596075   
PAYMENT             0.000000       0.000             13057.604660   
TRANSFER       986961.106232  493726.170            113966.567108   

                        
                median  
type                    
CASH_IN   2.543064e+05  
CASH_OUT  5.820766e-11  
DEBIT     5.127276e-11  
PAYMENT   9.482190e+03  
TRANSFER  1.164153e-10

In [372]:
model_df.groupby('type').apply(
    lambda x: (
        (x['oldbalanceDest'] == 0) &
        (x['newbalanceDest'] == 0)
    ).mean()
)

/tmp/ipykernel_58/2113499134.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  model_df.groupby('type').apply(


type
CASH_IN     0.114348
CASH_OUT    0.000719
DEBIT       0.000000
PAYMENT     1.000000
TRANSFER    0.007832
dtype: float64

In [373]:
numerical_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

In [374]:
model_df['hourly_volume_change'] = (
    model_df['hourly_volume_change'].fillna(0)
)

In [375]:
model_df[numerical_features].isnull().sum()

amount                      0
oldbalanceOrg               0
newbalanceOrig              0
orig_balance_change         0
orig_balance_discrepancy    0
amount_zscore_abs           0
dtype: int64

In [376]:
feature_columns = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs',
    'type'
]

X = model_df[feature_columns].copy()

In [377]:
X = pd.get_dummies(
    X,
    columns=['type'],
    dtype=int
)

In [378]:
X.shape

(6362620, 11)

In [379]:
X.columns.tolist()

['amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'orig_balance_change',
 'orig_balance_discrepancy',
 'amount_zscore_abs',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER']

In [380]:
y = model_df['isFraud'].copy()

In [381]:
y.value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [382]:
model_df = model_df.sort_values(
    by=['step', 'nameOrig', 'nameDest'],
    kind='mergesort'
).reset_index(drop=True)

In [383]:
train_df = model_df[model_df['step'] <= 355].copy()
test_df = model_df[model_df['step'] > 355].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("Training steps:", train_df['step'].min(), "to", train_df['step'].max())
print("Testing steps:", test_df['step'].min(), "to", test_df['step'].max())

Training shape: (5113884, 22)
Testing shape: (1248736, 22)
Training steps: 1 to 355
Testing steps: 356 to 743


In [384]:
print("Training labels:")
print(train_df['isFraud'].value_counts())

print("\nTesting labels:")
print(test_df['isFraud'].value_counts())

Training labels:
isFraud
0    5109921
1       3963
Name: count, dtype: int64

Testing labels:
isFraud
0    1244486
1       4250
Name: count, dtype: int64


In [385]:
train_type_stats = train_df.groupby('type')['amount'].agg(
    mean='mean',
    std='std'
)

train_type_stats

,mean,std
type,,
CASH_IN,168764.154603,1.265988e+05
CASH_OUT,176473.498231,1.561471e+05
DEBIT,5592.754116,1.262620e+04
PAYMENT,12854.768219,1.239073e+04
TRANSFER,930792.038517,2.023719e+06


In [386]:
train_df['type_amount_mean'] = (
    train_df['type'].map(train_type_stats['mean'])
)

train_df['type_amount_std'] = (
    train_df['type'].map(train_type_stats['std'])
)

train_df['amount_zscore_abs'] = (
    (
        train_df['amount'] - train_df['type_amount_mean']
    ) / train_df['type_amount_std']
).abs()

In [387]:
test_df['type_amount_mean'] = (
    test_df['type'].map(train_type_stats['mean'])
)

test_df['type_amount_std'] = (
    test_df['type'].map(train_type_stats['std'])
)

test_df['amount_zscore_abs'] = (
    (
        test_df['amount'] - test_df['type_amount_mean']
    ) / test_df['type_amount_std']
).abs()

In [388]:
print("Training:")
print(train_df['amount_zscore_abs'].describe())

print("\nTesting:")
print(test_df['amount_zscore_abs'].describe())

print("\nMissing values:")
print("Train:", train_df['amount_zscore_abs'].isnull().sum())
print("Test:", test_df['amount_zscore_abs'].isnull().sum())

Training:
count    5.113884e+06
mean     6.909754e-01
std      7.228776e-01
min      1.437048e-07
25%      2.813004e-01
50%      5.679472e-01
75%      9.129855e-01
max      6.291200e+01
Name: amount_zscore_abs, dtype: float64

Testing:
count    1.248736e+06
mean     7.039966e-01
std      9.808635e-01
min      3.088822e-07
25%      2.836719e-01
50%      5.686142e-01
75%      9.175604e-01
max      6.291200e+01
Name: amount_zscore_abs, dtype: float64

Missing values:
Train: 0
Test: 0


In [389]:
numerical_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

X_train = train_df[numerical_features + ['type']].copy()
X_test = test_df[numerical_features + ['type']].copy()

In [390]:
X_train = pd.get_dummies(
    X_train,
    columns=['type'],
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=['type'],
    dtype=int
)

In [391]:
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [392]:
y_train = train_df['isFraud'].copy()
y_test = test_df['isFraud'].copy()

In [393]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

print("\nMissing values:")
print("Train:", X_train.isnull().sum().sum())
print("Test:", X_test.isnull().sum().sum())

X_train shape: (5113884, 11)
X_test shape: (1248736, 11)

Features:
['amount', 'oldbalanceOrg', 'newbalanceOrig', 'orig_balance_change', 'orig_balance_discrepancy', 'amount_zscore_abs', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

Missing values:
Train: 0
Test: 0


In [394]:
from sklearn.ensemble import IsolationForest

In [395]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

In [396]:
iso_forest.fit(X_train)

IsolationForest(n_jobs=-1, random_state=42)

In [397]:
print("Isolation Forest training complete!")

Isolation Forest training complete!


In [398]:
test_scores = iso_forest.decision_function(X_test)

test_df['anomaly_score'] = test_scores

In [399]:
test_df['anomaly_score'].describe()

count    1.248736e+06
mean     7.126963e-02
std      6.453197e-02
min     -2.891450e-01
25%      3.734249e-02
50%      9.669826e-02
75%      1.210831e-01
max      1.358705e-01
Name: anomaly_score, dtype: float64

In [400]:
test_predictions = iso_forest.predict(X_test)

test_df['anomaly_prediction'] = test_predictions

In [401]:
test_df['anomaly_prediction'].value_counts()

anomaly_prediction
 1    1076575
-1     172161
Name: count, dtype: int64

In [402]:
anomaly_count = (
    test_df['anomaly_prediction'] == -1
).sum()

anomaly_percentage = (
    anomaly_count / len(test_df)
) * 100

print("Anomalies detected:", anomaly_count)
print("Percentage flagged:", anomaly_percentage)

Anomalies detected: 172161
Percentage flagged: 13.78682123363145


In [403]:
test_df['predicted_anomaly'] = (
    test_df['anomaly_prediction'] == -1
).astype(int)

In [404]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(
    test_df['isFraud'],
    test_df['predicted_anomaly']
)

print(cm)

[[1075364  169122]
 [   1211    3039]]


In [405]:
print(
    classification_report(
        test_df['isFraud'],
        test_df['predicted_anomaly'],
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9989    0.8641    0.9266   1244486
           1     0.0177    0.7151    0.0345      4250

    accuracy                         0.8636   1248736
   macro avg     0.5083    0.7896    0.4805   1248736
weighted avg     0.9955    0.8636    0.9236   1248736



In [406]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.00,
    -0.02,
    -0.04,
    -0.06,
    -0.08,
    -0.10,
    -0.12,
    -0.14,
    -0.16,
    -0.18,
    -0.20
]

results = []

for threshold in thresholds:
    predictions = (
        test_df['anomaly_score'] < threshold
    ).astype(int)

    precision = precision_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        test_df['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    results.append({
        'threshold': threshold,
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results = pd.DataFrame(results)

threshold_results

,threshold,flagged,flagged_percentage,precision,recall,f1
0,0.00,172161,13.786821,0.017652,0.715059,0.034454
1,-0.02,124220,9.947659,0.022017,0.643529,0.042578
2,-0.04,83256,6.667222,0.031397,0.615059,0.059744
3,-0.06,58261,4.665598,0.039632,0.543294,0.073875
4,-0.08,41202,3.299496,0.050216,0.486824,0.091041
5,-0.10,30295,2.426053,0.062651,0.446588,0.109886
6,-0.12,22202,1.777958,0.071030,0.371059,0.119235
7,-0.14,15609,1.249984,0.077840,0.285882,0.122363
8,-0.16,10719,0.858388,0.090400,0.228000,0.129468
9,-0.18,5045,0.404009,0.167493,0.198824,0.181818


In [407]:
validation_df = test_df[
    test_df['step'] <= 549
].copy()

final_test_df = test_df[
    test_df['step'] >= 550
].copy()

In [408]:
print("Validation shape:", validation_df.shape)
print(
    "Validation steps:",
    validation_df['step'].min(),
    "to",
    validation_df['step'].max()
)

print("\nFinal test shape:", final_test_df.shape)
print(
    "Final test steps:",
    final_test_df['step'].min(),
    "to",
    final_test_df['step'].max()
)

print("\nValidation labels:")
print(validation_df['isFraud'].value_counts())

print("\nFinal test labels:")
print(final_test_df['isFraud'].value_counts())

Validation shape: (1047331, 27)
Validation steps: 356 to 549

Final test shape: (201405, 27)
Final test steps: 550 to 743

Validation labels:
isFraud
0    1045237
1       2094
Name: count, dtype: int64

Final test labels:
isFraud
0    199249
1      2156
Name: count, dtype: int64


In [409]:
X_validation = validation_df[
    numerical_features + ['type']
].copy()

X_validation = pd.get_dummies(
    X_validation,
    columns=['type'],
    dtype=int
)

X_validation = X_validation.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Validation shape:", X_validation.shape)
print("Missing values:", X_validation.isnull().sum().sum())

Validation shape: (1047331, 11)
Missing values: 0


In [410]:
validation_scores = iso_forest.decision_function(X_validation)

validation_df['anomaly_score'] = validation_scores

validation_df['anomaly_score'].describe()

count    1.047331e+06
mean     7.091714e-02
std      6.527985e-02
min     -2.891450e-01
25%      3.706298e-02
50%      9.676414e-02
75%      1.211847e-01
max      1.358705e-01
Name: anomaly_score, dtype: float64

In [411]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(-0.12, -0.221, -0.01)

validation_results = []

for threshold in thresholds:

    predictions = (
        validation_df['anomaly_score'] < threshold
    ).astype(int)

    precision = precision_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        validation_df['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    validation_results.append({
        'threshold': round(threshold, 2),
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

validation_threshold_results = pd.DataFrame(
    validation_results
)

validation_threshold_results

,threshold,flagged,flagged_percentage,precision,recall,f1
0,-0.12,19903,1.900354,0.040195,0.382044,0.072737
1,-0.13,16770,1.601213,0.042993,0.344317,0.076442
2,-0.14,14138,1.349908,0.044561,0.300860,0.077624
3,-0.15,12192,1.164102,0.046178,0.268863,0.078818
4,-0.16,9708,0.926928,0.051813,0.240210,0.085240
5,-0.17,6888,0.657672,0.066202,0.217765,0.101536
6,-0.18,4326,0.413050,0.097550,0.201528,0.131464
7,-0.19,2519,0.240516,0.150457,0.180993,0.164318
8,-0.20,1446,0.138065,0.201245,0.138968,0.164407
9,-0.21,871,0.083164,0.251435,0.104585,0.147723


In [412]:
best_row = validation_threshold_results.loc[
    validation_threshold_results['f1'].idxmax()
]

best_row

threshold               -0.200000
flagged               1446.000000
flagged_percentage       0.138065
precision                0.201245
recall                   0.138968
f1                       0.164407
Name: 8, dtype: float64

In [413]:
X_final_test = final_test_df[
    numerical_features + ['type']
].copy()

X_final_test = pd.get_dummies(
    X_final_test,
    columns=['type'],
    dtype=int
)

X_final_test = X_final_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Final test shape:", X_final_test.shape)
print("Missing values:", X_final_test.isnull().sum().sum())

Final test shape: (201405, 11)
Missing values: 0


In [414]:
final_scores = iso_forest.decision_function(X_final_test)

final_test_df['anomaly_score'] = final_scores

In [415]:
selected_threshold = -0.18

final_test_df['predicted_anomaly'] = (
    final_test_df['anomaly_score'] < selected_threshold
).astype(int)

In [416]:
from sklearn.metrics import confusion_matrix, classification_report

final_cm = confusion_matrix(
    final_test_df['isFraud'],
    final_test_df['predicted_anomaly']
)

print("Confusion Matrix:")
print(final_cm)

Confusion Matrix:
[[198953    296]
 [  1733    423]]


In [417]:
print(
    classification_report(
        final_test_df['isFraud'],
        final_test_df['predicted_anomaly'],
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9914    0.9985    0.9949    199249
           1     0.5883    0.1962    0.2943      2156

    accuracy                         0.9899    201405
   macro avg     0.7898    0.5974    0.6446    201405
weighted avg     0.9871    0.9899    0.9874    201405



In [418]:
final_flagged = final_test_df['predicted_anomaly'].sum()

final_flagged_percentage = (
    final_flagged / len(final_test_df)
) * 100

print("Total final-test transactions:", len(final_test_df))
print("Anomalies flagged:", final_flagged)
print("Percentage flagged:", final_flagged_percentage)

Total final-test transactions: 201405
Anomalies flagged: 719
Percentage flagged: 0.35699213028474963


# Model 2: Isolation Forest Improvement

## Analysis of Model 1 Errors

Model 1 achieved a precision of 58.16%, recall of 20.83%, and F1-score
of 30.67% on the final chronological test set using an anomaly threshold
of -0.18.

Although the model produced relatively few false-positive alerts, its
recall indicates that a substantial proportion of labelled fraudulent
transactions were not detected. Model 2 therefore investigates the
characteristics of detected and missed fraudulent transactions to guide
feature and model improvements.

## Analysis of Model 1 Errors

Before developing the second model, the fraudulent transactions correctly
detected by Model 1 are compared with fraudulent transactions that were
missed. This analysis helps identify transaction characteristics that may
improve the anomaly detection model.

In [419]:
# Fraud transactions correctly detected by Model 1
detected_fraud = final_test_df[
    (final_test_df['isFraud'] == 1) &
    (final_test_df['predicted_anomaly'] == 1)
].copy()

# Fraud transactions missed by Model 1
missed_fraud = final_test_df[
    (final_test_df['isFraud'] == 1) &
    (final_test_df['predicted_anomaly'] == 0)
].copy()

print("Detected fraud:", len(detected_fraud))
print("Missed fraud:", len(missed_fraud))

Detected fraud: 423
Missed fraud: 1733


In [420]:
print("Detected fraud by transaction type:")
print(detected_fraud['type'].value_counts())

print("\nMissed fraud by transaction type:")
print(missed_fraud['type'].value_counts())

Detected fraud by transaction type:
type
CASH_OUT    312
TRANSFER    111
Name: count, dtype: int64

Missed fraud by transaction type:
type
TRANSFER    967
CASH_OUT    766
Name: count, dtype: int64


In [421]:
print("Detected fraud percentages:")
print(
    detected_fraud['type']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissed fraud percentages:")
print(
    missed_fraud['type']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Detected fraud percentages:
type
CASH_OUT    73.76
TRANSFER    26.24
Name: proportion, dtype: float64

Missed fraud percentages:
type
TRANSFER    55.8
CASH_OUT    44.2
Name: proportion, dtype: float64


From the above results, we can see that Model 1 disproportionately catches CASH_OUT fraud, while TRANSFER makes up the majority of the fraud it misses.

In [422]:
comparison_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

detected_summary = detected_fraud[comparison_features].median()
missed_summary = missed_fraud[comparison_features].median()

fraud_comparison = pd.DataFrame({
    'Detected Fraud Median': detected_summary,
    'Missed Fraud Median': missed_summary
})

fraud_comparison

,Detected Fraud Median,Missed Fraud Median
amount,5.566368e+06,283121.680000
oldbalanceOrg,5.566368e+06,283121.680000
newbalanceOrig,0.000000e+00,0.000000
orig_balance_change,5.101856e+06,281087.010000
orig_balance_discrepancy,0.000000e+00,0.000000
amount_zscore_abs,1.202813e+01,0.432969


From the above analysis, we see that Model 1 is very good at noticing extremely large fraudulent transactions, but much worse when a fraudulent transaction has an amount that looks more ordinary. This is apart from newbalanceOrig and orig_balance_discrepancy.

- Remember, **amount_zscore_abs** measures approximately: "How unusual is this transaction amount compared with typical transactions of the same transaction type?"

In [423]:
print("Detected fraud anomaly scores:")
print(detected_fraud['anomaly_score'].describe())

print("\nMissed fraud anomaly scores:")
print(missed_fraud['anomaly_score'].describe())

Detected fraud anomaly scores:
count    423.000000
mean      -0.214020
std        0.024148
min       -0.289145
25%       -0.224880
50%       -0.207884
75%       -0.196954
max       -0.180329
Name: anomaly_score, dtype: float64

Missed fraud anomaly scores:
count    1733.000000
mean       -0.033924
std         0.081575
min        -0.179090
25%        -0.105546
50%        -0.040691
75%         0.022390
max         0.108377
Name: anomaly_score, dtype: float64


From the above results we see that the detected fraud have anomaly scores close to our threshold (-0.20) while our missed ones have anomaly scores that are very far from -0.20, (about −0.041 -> above the median). Isolation Forest considers many of these transactions considerably more normal than the fraud it successfully detects.
So simply changing:selected_threshold = -0.20 to something less strict would likely also introduce a large number of normal transactions as false positives. We already saw that behaviour during threshold testing. We need to give Isolation Forest additional features that help identify unusual transaction behaviour even when the transaction amount itself isn't extreme.

How we are gonna go about this is to express transaction behaviour relative to the origin account's available balance. Model 1 uses raw balances, but ratios can reveal structurally unusual transactions even when the absolute amount is not huge.

In [424]:
import numpy as np

model_df['amount_to_oldbalance_ratio'] = np.where(
    model_df['oldbalanceOrg'] > 0,
    model_df['amount'] / model_df['oldbalanceOrg'],
    0
)

In [425]:
model_df['balance_change_ratio'] = np.where(
    model_df['oldbalanceOrg'] > 0,
    model_df['orig_balance_change'] / model_df['oldbalanceOrg'],
    0
)

Sth you might be wondering is why we are using amount and orig_balance_change to calculate different ratios cz aren't they the same? They are not. For transfers they happen to be the same but when there's a CASH_IN operation, the orig_balance_change becomes negative. So the ratios might have the same values numerically but different signs(orig_balance_change = oldbalanceOrg - newbalanceOrig).

In [426]:
model_df['origin_balance_zero_after'] = (
    model_df['newbalanceOrig'] == 0
).astype(int)

# Result:
# 1 = balance is zero after transaction
# 0 = balance is not zero after transaction

In [427]:
new_features = [
    'amount_to_oldbalance_ratio',
    'balance_change_ratio',
    'origin_balance_zero_after'
]

model_df[new_features].describe()

,amount_to_oldbalance_ratio,balance_change_ratio,origin_balance_zero_after
count,6.362620e+06,6.362620e+06,6.362620e+06
mean,8.171346e+01,-1.589650e+01,5.673081e-01
std,3.395705e+03,8.961777e+02,4.954489e-01
min,0.000000e+00,-5.489098e+05,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,7.681118e-02,0.000000e+00,1.000000e+00
75%,2.219448e+00,8.358802e-01,1.000000e+00
max,3.925476e+06,1.000000e+00,1.000000e+00


In [428]:
model_df.groupby('isFraud')[new_features].median()

,amount_to_oldbalance_ratio,balance_change_ratio,origin_balance_zero_after
isFraud,,,
0,0.076139,0.0,1.0
1,1.000000,1.0,1.0


The above results show us that for the typical **fraud transaction**:

amount = entire old balance

and:

balance change = entire old balance

So both ratios are exactly:
1.0

This is actually a very strong PaySim fraud pattern.
It tells us that many PaySim-labelled fraud transactions drain the entire origin balance.
And this is exactly why we need to be careful: this feature could make our PaySim evaluation much better, but it may also be a dataset-specific pattern rather than something that generalizes directly to elderly-care billing transactions.
We can still experiment with it for Model 2, but we'll document this limitation.

In [429]:
print(
    model_df[
        [
            'amount',
            'oldbalanceOrg',
            'newbalanceOrig',
            'amount_to_oldbalance_ratio',
            'balance_change_ratio',
            'origin_balance_zero_after',
            'isFraud'
        ]
    ].head(10)
)

      amount  oldbalanceOrg  newbalanceOrig  amount_to_oldbalance_ratio  \
0    2655.04       21214.00        18558.96                    0.125155   
1    2444.49       31454.00        29009.51                    0.077716   
2    1246.74        1468.00          221.26                    0.849278   
3     871.75       19869.00        18997.25                    0.043875   
4     264.93           0.00            0.00                    0.000000   
5    9029.12       25480.00        16450.88                    0.354361   
6    1617.90      507865.00       506247.10                    0.003186   
7   59390.46           0.00            0.00                    0.000000   
8    1213.64       86078.00        84864.36                    0.014099   
9  377287.80     5110945.41      5488233.21                    0.073820   

   balance_change_ratio  origin_balance_zero_after  isFraud  
0              0.125155                          0        0  
1              0.077716                          0

In [430]:
print(
    model_df.groupby('isFraud')['origin_balance_zero_after']
    .agg(['mean', 'median', 'sum', 'count'])
)

             mean  median      sum    count
isFraud                                    
0        0.566774     1.0  3601513  6354407
1        0.980519     1.0     8053     8213


In [431]:
print("Amount-to-balance ratio percentiles:")

print(
    model_df.loc[
        model_df['oldbalanceOrg'] > 0,
        'amount_to_oldbalance_ratio'
    ].quantile([
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999
    ])
)

Amount-to-balance ratio percentiles:
0.500        0.740758
0.750        6.655752
0.900       39.487901
0.950      176.531520
0.990     1675.171203
0.999    14164.512202
Name: amount_to_oldbalance_ratio, dtype: float64


In [432]:
model_df['log_amount_to_oldbalance_ratio'] = np.log1p(
    model_df['amount_to_oldbalance_ratio']
)

In [433]:
model_df['log_amount_to_oldbalance_ratio'].describe()

count    6.362620e+06
mean     8.898274e-01
std      1.553307e+00
min      0.000000e+00
25%      0.000000e+00
50%      7.400406e-02
75%      1.169210e+00
max      1.518300e+01
Name: log_amount_to_oldbalance_ratio, dtype: float64

In [434]:
model_df.groupby('isFraud')[
    'log_amount_to_oldbalance_ratio'
].median()

isFraud
0    0.073380
1    0.693147
Name: log_amount_to_oldbalance_ratio, dtype: float64

In [435]:
# Fresh chronological splits for Model 2
train_df_model2 = model_df[
    model_df['step'] <= 355
].copy()

validation_df_model2 = model_df[
    (model_df['step'] >= 356) &
    (model_df['step'] <= 549)
].copy()

final_test_df_model2 = model_df[
    model_df['step'] >= 550
].copy()

print("Training:", train_df_model2.shape)
print("Validation:", validation_df_model2.shape)
print("Final test:", final_test_df_model2.shape)

print("\nTraining steps:",
      train_df_model2['step'].min(),
      "to",
      train_df_model2['step'].max())

print("Validation steps:",
      validation_df_model2['step'].min(),
      "to",
      validation_df_model2['step'].max())

print("Final test steps:",
      final_test_df_model2['step'].min(),
      "to",
      final_test_df_model2['step'].max())

Training: (5113884, 26)
Validation: (1047331, 26)
Final test: (201405, 26)

Training steps: 1 to 355
Validation steps: 356 to 549
Final test steps: 550 to 743


In [436]:
train_type_stats_model2 = (
    train_df_model2
    .groupby('type')['amount']
    .agg(mean='mean', std='std')
)

for dataset in [
    train_df_model2,
    validation_df_model2,
    final_test_df_model2
]:
    dataset['type_amount_mean'] = dataset['type'].map(
        train_type_stats_model2['mean']
    )

    dataset['type_amount_std'] = dataset['type'].map(
        train_type_stats_model2['std']
    )

    dataset['amount_zscore_abs'] = (
        (
            dataset['amount'] -
            dataset['type_amount_mean']
        ) /
        dataset['type_amount_std']
    ).abs()

In [437]:
numerical_features_model2 = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs',
    'log_amount_to_oldbalance_ratio'
]

In [438]:
X_train_model2 = train_df_model2[
    numerical_features_model2 + ['type']
].copy()

X_validation_model2 = validation_df_model2[
    numerical_features_model2 + ['type']
].copy()

X_train_model2 = pd.get_dummies(
    X_train_model2,
    columns=['type'],
    dtype=int
)

X_validation_model2 = pd.get_dummies(
    X_validation_model2,
    columns=['type'],
    dtype=int
)

X_validation_model2 = X_validation_model2.reindex(
    columns=X_train_model2.columns,
    fill_value=0
)

print("Model 2 training shape:", X_train_model2.shape)
print("Model 2 validation shape:", X_validation_model2.shape)

print("Training missing values:",
      X_train_model2.isnull().sum().sum())

print("Validation missing values:",
      X_validation_model2.isnull().sum().sum())

Model 2 training shape: (5113884, 12)
Model 2 validation shape: (1047331, 12)
Training missing values: 0
Validation missing values: 0


In [439]:
from sklearn.ensemble import IsolationForest

iso_forest_model2 = IsolationForest(
    n_estimators=100,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest_model2.fit(X_train_model2)

print("Model 2 training complete.")

Model 2 training complete.


In [440]:
validation_scores_model2 = (
    iso_forest_model2.decision_function(
        X_validation_model2
    )
)

validation_df_model2['anomaly_score'] = (
    validation_scores_model2
)

validation_df_model2['anomaly_score'].describe()

count    1.047331e+06
mean     6.695708e-02
std      6.319771e-02
min     -2.769598e-01
25%      3.272977e-02
50%      8.664512e-02
75%      1.175324e-01
max      1.359517e-01
Name: anomaly_score, dtype: float64

In [441]:
thresholds = np.arange(-0.12, -0.221, -0.01)

validation_results_model2 = []

for threshold in thresholds:

    predictions = (
        validation_df_model2['anomaly_score'] < threshold
    ).astype(int)

    precision = precision_score(
        validation_df_model2['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        validation_df_model2['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        validation_df_model2['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    validation_results_model2.append({
        'threshold': round(threshold, 2),
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

validation_threshold_results_model2 = pd.DataFrame(
    validation_results_model2
)

validation_threshold_results_model2

,threshold,flagged,flagged_percentage,precision,recall,f1
0,-0.12,16075,1.534854,0.042986,0.329990,0.076064
1,-0.13,12220,1.166775,0.054828,0.319962,0.093615
2,-0.14,9301,0.888067,0.063649,0.282713,0.103905
3,-0.15,7084,0.676386,0.075663,0.255969,0.116801
4,-0.16,5283,0.504425,0.074768,0.188634,0.107090
5,-0.17,3690,0.352324,0.077778,0.137058,0.099239
6,-0.18,2572,0.245577,0.083981,0.103152,0.092585
7,-0.19,1707,0.162986,0.113064,0.092168,0.101552
8,-0.20,1118,0.106748,0.115385,0.061605,0.080324
9,-0.21,667,0.063686,0.085457,0.027221,0.041289


In [442]:
best_row_model2 = (
    validation_threshold_results_model2.loc[
        validation_threshold_results_model2['f1'].idxmax()
    ]
)

best_row_model2

threshold               -0.150000
flagged               7084.000000
flagged_percentage       0.676386
precision                0.075663
recall                   0.255969
f1                       0.116801
Name: 3, dtype: float64

Model 2's recall increased from 13.90% → 25.60%. In other words, the new ratio feature helped Isolation Forest identify more of the fraud-labelled transactions as unusual.
But precision collapsed from 20.12% → 7.57%. It started identifying a lot more normal transactions as unusual too.

And the alert volume jumped:
- Model 1: 1,446 alerts
- Model 2: 7,084 alerts

That's nearly 5× as many alerts.
As a result, F1 fell:
- Model 1: 0.1644
- Model 2: 0.1168

Therefore, under our predetermined validation F1 criterion, this version of Model 2 does not improve on Model 1.

However, we've learnt sth. log_amount_to_oldbalance_ratio makes the model more sensitive to fraud-like patterns, which explains the recall improvement.
But the pattern isn't sufficiently specific to fraud.

In [443]:
fraud_type_analysis = (
    final_test_df[
        final_test_df['isFraud'] == 1
    ]
    .groupby('type')
    [
        [
            'amount',
            'oldbalanceOrg',
            'newbalanceOrig',
            'orig_balance_change',
            'orig_balance_discrepancy',
            'amount_zscore_abs'
        ]
    ]
    .median()
)

fraud_type_analysis

,amount,oldbalanceOrg,newbalanceOrig,orig_balance_change,orig_balance_discrepancy,amount_zscore_abs
type,,,,,,
CASH_OUT,417116.395,417116.395,0.0,417116.395,0.0,1.541129
TRANSFER,436270.645,438267.230,0.0,417116.395,0.0,0.386822


In [444]:
fraud_detection_by_type = (
    final_test_df[
        final_test_df['isFraud'] == 1
    ]
    .groupby('type')['predicted_anomaly']
    .agg(
        total_fraud='count',
        detected='sum'
    )
)

fraud_detection_by_type['detection_rate'] = (
    fraud_detection_by_type['detected']
    /
    fraud_detection_by_type['total_fraud']
) * 100

fraud_detection_by_type

,total_fraud,detected,detection_rate
type,,,
CASH_OUT,1078,312,28.942486
TRANSFER,1078,111,10.296846


Rn we're asking one Isolation Forest to learn all five transaction types simultaneously ( CASH_IN ,CASH_OUT ,DEBIT, PAYMENT, TRANSFER ) even though their normal behaviour is very different.

We already know, for example, that typical TRANSFER amounts are much larger than typical PAYMENT amounts. We're trying to help the model understand this with one-hot transaction types and amount_zscore_abs, but Isolation Forest is still constructing its trees across the entire mixed dataset.

What if Isolation Forest learns normal behaviour separately for each transaction type? So basically one Isolation Forest for each payment type?

In [445]:
validation_type_analysis = (
    validation_df[
        validation_df['isFraud'] == 0
    ]
    .groupby('type')
    [
        [
            'amount',
            'oldbalanceOrg',
            'newbalanceOrig',
            'orig_balance_change',
            'orig_balance_discrepancy'
        ]
    ]
    .median()
)

validation_type_analysis

,amount,oldbalanceOrg,newbalanceOrig,orig_balance_change,orig_balance_discrepancy
type,,,,,
CASH_IN,144688.360,1274171.14,1447117.690,-144688.36,2.893767e+05
CASH_OUT,143076.585,575.00,0.000,560.00,1.167809e+05
DEBIT,2794.825,20918.50,17644.955,1845.70,2.728484e-12
PAYMENT,10563.070,10154.00,0.000,2149.82,9.734900e+02
TRANSFER,501721.390,0.00,0.000,0.00,4.701019e+05


The above output answers: How different does "normal" look for each transaction type?

In [446]:
validation_type_counts = (
    validation_df[
        validation_df['isFraud'] == 0
    ]['type']
    .value_counts()
)

validation_type_counts

type
PAYMENT     360961
CASH_OUT    357880
CASH_IN     230879
TRANSFER     88235
DEBIT         7282
Name: count, dtype: int64

The above output answers: Do we actually have enough normal training/validation examples of each type to justify type-specific modelling?

## Model 2B: Type-Specific Isolation Forests

The validation analysis showed substantial differences in the normal
transaction distributions of CASH_IN, CASH_OUT, DEBIT, PAYMENT and
TRANSFER transactions. A single Isolation Forest may therefore have
difficulty representing normal behaviour across all transaction types.

In this experiment, a separate Isolation Forest is trained for each
transaction type. Each transaction is consequently evaluated relative
to transactions of the same type rather than against the entire
transaction population.

So far:<br>
- Model 1  = One model + original features
- Model 2A = One model + ratio feature
- Model 2B = Separate models + original features

In [447]:
numerical_features_model2b = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

In [448]:
transaction_types = [
    'CASH_IN',
    'CASH_OUT',
    'DEBIT',
    'PAYMENT',
    'TRANSFER'
]

models_model2b = {}

In [449]:
from sklearn.ensemble import IsolationForest

for transaction_type in transaction_types:

    type_train = train_df[
        train_df['type'] == transaction_type
    ]

    X_type_train = type_train[
        numerical_features_model2b
    ]

    model = IsolationForest(
        n_estimators=100,
        contamination='auto',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_type_train)

    models_model2b[transaction_type] = model

    print(
        transaction_type,
        "- training rows:",
        len(X_type_train)
    )

CASH_IN - training rows: 1122474
CASH_OUT - training rows: 1813999
DEBIT - training rows: 32533
PAYMENT - training rows: 1720408
TRANSFER - training rows: 424470


In [450]:
validation_model2b = validation_df.copy()

validation_model2b['anomaly_score_model2b'] = np.nan

In [451]:
for transaction_type in transaction_types:

    mask = (
        validation_model2b['type']
        == transaction_type
    )

    X_type_validation = validation_model2b.loc[
        mask,
        numerical_features_model2b
    ]

    scores = models_model2b[
        transaction_type
    ].decision_function(
        X_type_validation
    )

    validation_model2b.loc[
        mask,
        'anomaly_score_model2b'
    ] = scores

In [452]:
print(
    "Missing scores:",
    validation_model2b[
        'anomaly_score_model2b'
    ].isnull().sum()
)

validation_model2b[
    'anomaly_score_model2b'
].describe()

Missing scores: 0


count    1.047331e+06
mean     6.616976e-02
std      7.620753e-02
min     -3.455285e-01
25%      3.632579e-02
50%      8.771879e-02
75%      1.208149e-01
max      1.676671e-01
Name: anomaly_score_model2b, dtype: float64

In [453]:
score_distribution_by_type = (
    validation_model2b
    .groupby('type')[
        'anomaly_score_model2b'
    ]
    .describe()
)

score_distribution_by_type

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
CASH_IN,230879.0,0.029210,0.063255,-0.305871,0.002055,0.049035,0.075459,0.104735
CASH_OUT,358927.0,0.075268,0.073440,-0.292891,0.055612,0.103922,0.124015,0.143534
DEBIT,7282.0,0.106537,0.071853,-0.345528,0.101379,0.130748,0.143540,0.158108
PAYMENT,360961.0,0.068848,0.076395,-0.300437,0.045464,0.100511,0.120767,0.136618
TRANSFER,89282.0,0.111050,0.077715,-0.332473,0.101883,0.141705,0.159730,0.167667


In [454]:
validation_model2b.groupby(
    ['type', 'isFraud']
)['anomaly_score_model2b'].median()

type      isFraud
CASH_IN   0          0.049035
CASH_OUT  0          0.104096
          1         -0.218126
DEBIT     0          0.130748
PAYMENT   0          0.100511
TRANSFER  0          0.142058
          1         -0.092489
Name: anomaly_score_model2b, dtype: float64

In [455]:
thresholds_model2b = np.arange(
    -0.05,
    -0.251,
    -0.01
)

In [456]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

validation_results_model2b = []

for threshold in thresholds_model2b:

    predictions = (
        validation_model2b[
            'anomaly_score_model2b'
        ] < threshold
    ).astype(int)

    precision = precision_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    flagged_percentage = (
        flagged / len(predictions)
    ) * 100

    validation_results_model2b.append({
        'threshold': round(threshold, 2),
        'flagged': flagged,
        'flagged_percentage': flagged_percentage,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

validation_threshold_results_model2b = pd.DataFrame(
    validation_results_model2b
)

validation_threshold_results_model2b

,threshold,flagged,flagged_percentage,precision,recall,f1
0,-0.05,95177,9.087576,0.013848,0.629417,0.027100
1,-0.06,85617,8.174780,0.014997,0.613181,0.029278
2,-0.07,76565,7.310487,0.016457,0.601719,0.032037
3,-0.08,68308,6.522102,0.018182,0.593123,0.035283
4,-0.09,59985,5.727416,0.020055,0.574499,0.038757
5,-0.10,51657,4.932252,0.022920,0.565425,0.044055
6,-0.11,45309,4.326139,0.025536,0.552531,0.048815
7,-0.12,38903,3.714489,0.029509,0.548233,0.056004
8,-0.13,33398,3.188868,0.033804,0.539160,0.063620
9,-0.14,28416,2.713182,0.038007,0.515759,0.070796


In [457]:
best_row_model2b = (
    validation_threshold_results_model2b.loc[
        validation_threshold_results_model2b[
            'f1'
        ].idxmax()
    ]
)

best_row_model2b

threshold               -0.250000
flagged               1712.000000
flagged_percentage       0.163463
precision                0.356308
recall                   0.291309
f1                       0.320547
Name: 20, dtype: float64

In [458]:
selected_threshold_model2b = best_row_model2b[
    'threshold'
]

validation_model2b[
    'predicted_anomaly_model2b'
] = (
    validation_model2b[
        'anomaly_score_model2b'
    ] < selected_threshold_model2b
).astype(int)

In [459]:
alerts_by_type_model2b = (
    validation_model2b
    .groupby('type')
    .agg(
        total_transactions=(
            'predicted_anomaly_model2b',
            'count'
        ),
        flagged=(
            'predicted_anomaly_model2b',
            'sum'
        )
    )
)

alerts_by_type_model2b[
    'flagged_percentage'
] = (
    alerts_by_type_model2b['flagged']
    /
    alerts_by_type_model2b['total_transactions']
) * 100

alerts_by_type_model2b

,total_transactions,flagged,flagged_percentage
type,,,
CASH_IN,230879,570,0.246883
CASH_OUT,358927,526,0.146548
DEBIT,7282,72,0.988739
PAYMENT,360961,366,0.101396
TRANSFER,89282,178,0.199368


In [460]:
fraud_detection_model2b = (
    validation_model2b[
        validation_model2b['isFraud'] == 1
    ]
    .groupby('type')[
        'predicted_anomaly_model2b'
    ]
    .agg(
        total_fraud='count',
        detected='sum'
    )
)

fraud_detection_model2b[
    'detection_rate'
] = (
    fraud_detection_model2b['detected']
    /
    fraud_detection_model2b['total_fraud']
) * 100

fraud_detection_model2b

,total_fraud,detected,detection_rate
type,,,
CASH_OUT,1047,465,44.412607
TRANSFER,1047,145,13.849093


In [461]:
thresholds_model2b_extended = np.arange(
    -0.20,
    -0.351,
    -0.01
)

extended_results_model2b = []

for threshold in thresholds_model2b_extended:

    predictions = (
        validation_model2b[
            'anomaly_score_model2b'
        ] < threshold
    ).astype(int)

    precision = precision_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        validation_model2b['isFraud'],
        predictions,
        zero_division=0
    )

    flagged = predictions.sum()

    extended_results_model2b.append({
        'threshold': round(threshold, 2),
        'flagged': flagged,
        'flagged_percentage':
            flagged / len(predictions) * 100,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

extended_threshold_results_model2b = pd.DataFrame(
    extended_results_model2b
)

extended_threshold_results_model2b

,threshold,flagged,flagged_percentage,precision,recall,f1
0,-0.20,8296,0.792109,0.099928,0.395893,0.159577
1,-0.21,6603,0.630460,0.119340,0.376313,0.181212
2,-0.22,5016,0.478932,0.151914,0.363897,0.214346
3,-0.23,3630,0.346595,0.201377,0.349093,0.255416
4,-0.24,2420,0.231064,0.279752,0.323305,0.299956
5,-0.25,1712,0.163463,0.356308,0.291309,0.320547
6,-0.26,1127,0.107607,0.518190,0.278892,0.362620
7,-0.27,757,0.072279,0.624835,0.225883,0.331813
8,-0.28,524,0.050032,0.683206,0.170965,0.273491
9,-0.29,280,0.026735,0.678571,0.090735,0.160067


In [462]:
best_extended_model2b = (
    extended_threshold_results_model2b.loc[
        extended_threshold_results_model2b[
            'f1'
        ].idxmax()
    ]
)

best_extended_model2b

threshold               -0.260000
flagged               1127.000000
flagged_percentage       0.107607
precision                0.518190
recall                   0.278892
f1                       0.362620
Name: 6, dtype: float64

In [463]:
selected_threshold_model2b = -0.26

print(
    "Locked Model 2B threshold:",
    selected_threshold_model2b
)

Locked Model 2B threshold: -0.26


In [464]:
final_model2b = final_test_df.copy()

final_model2b['anomaly_score_model2b'] = np.nan

for transaction_type in transaction_types:

    mask = (
        final_model2b['type']
        == transaction_type
    )

    X_type_final = final_model2b.loc[
        mask,
        numerical_features_model2b
    ]

    scores = models_model2b[
        transaction_type
    ].decision_function(
        X_type_final
    )

    final_model2b.loc[
        mask,
        'anomaly_score_model2b'
    ] = scores

print(
    "Missing final scores:",
    final_model2b[
        'anomaly_score_model2b'
    ].isnull().sum()
)

Missing final scores: 0


In [465]:
final_model2b[
    'predicted_anomaly_model2b'
] = (
    final_model2b[
        'anomaly_score_model2b'
    ] < selected_threshold_model2b
).astype(int)

In [466]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

y_final_model2b = final_model2b['isFraud']

predictions_final_model2b = final_model2b[
    'predicted_anomaly_model2b'
]

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_final_model2b,
        predictions_final_model2b
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_final_model2b,
        predictions_final_model2b,
        digits=4,
        zero_division=0
    )
)

print(
    "Precision:",
    precision_score(
        y_final_model2b,
        predictions_final_model2b,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_final_model2b,
        predictions_final_model2b,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_final_model2b,
        predictions_final_model2b,
        zero_division=0
    )
)

flagged_final_model2b = predictions_final_model2b.sum()

print(
    "Flagged transactions:",
    flagged_final_model2b
)

print(
    "Flagged percentage:",
    flagged_final_model2b
    / len(predictions_final_model2b)
    * 100
)

Confusion Matrix:
[[199193     56]
 [  1600    556]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9920    0.9997    0.9959    199249
           1     0.9085    0.2579    0.4017      2156

    accuracy                         0.9918    201405
   macro avg     0.9503    0.6288    0.6988    201405
weighted avg     0.9911    0.9918    0.9895    201405

Precision: 0.9084967320261438
Recall: 0.25788497217068646
F1: 0.40173410404624277
Flagged transactions: 612
Flagged percentage: 0.30386534594473824


In [467]:
final_detection_by_type_model2b = (
    final_model2b[
        final_model2b['isFraud'] == 1
    ]
    .groupby('type')[
        'predicted_anomaly_model2b'
    ]
    .agg(
        total_fraud='count',
        detected='sum'
    )
)

final_detection_by_type_model2b[
    'detection_rate'
] = (
    final_detection_by_type_model2b['detected']
    /
    final_detection_by_type_model2b['total_fraud']
) * 100

final_detection_by_type_model2b

,total_fraud,detected,detection_rate
type,,,
CASH_OUT,1078,435,40.352505
TRANSFER,1078,121,11.224490


## Model 2B Final Evaluation

Model 2B used separate Isolation Forest models for each transaction
type. The approach was motivated by validation analysis showing
substantial differences in the normal transaction distributions of
CASH_IN, CASH_OUT, DEBIT, PAYMENT and TRANSFER transactions.

A common anomaly threshold of -0.26 was selected using the validation
period based on the highest validation F1-score. The threshold was then
fixed before evaluation on the final chronological evaluation period.

On the final evaluation period, Model 2B achieved a precision of
90.85%, recall of 25.79%, and F1-score of 40.17%. The model flagged
612 of 201,405 transactions (0.304%). Of the 2,156 fraud-labelled
transactions, 556 were detected while 1,600 were missed.

Compared with the baseline Model 1, Model 2B increased precision,
recall, and F1-score while reducing the total number of transactions
flagged for review. However, recall remained limited, particularly for
TRANSFER transactions. Therefore, the model should be interpreted as
an anomaly-screening mechanism that identifies unusual transactions
for administrator review rather than as a definitive fraud
classification system.

In [468]:
# Final selected anomaly-detection configuration

FINAL_MODEL = "Model 2B - Type-Specific Isolation Forests"
FINAL_THRESHOLD = -0.26

FINAL_NUMERICAL_FEATURES = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'orig_balance_change',
    'orig_balance_discrepancy',
    'amount_zscore_abs'
]

print("Selected model:", FINAL_MODEL)
print("Selected threshold:", FINAL_THRESHOLD)
print("Features:", FINAL_NUMERICAL_FEATURES)

Selected model: Model 2B - Type-Specific Isolation Forests
Selected threshold: -0.26
Features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'orig_balance_change', 'orig_balance_discrepancy', 'amount_zscore_abs']


In [469]:
import os

model_output_dir = "/kaggle/working/models"
os.makedirs(model_output_dir, exist_ok=True)

print("Model directory created:", model_output_dir)

Model directory created: /kaggle/working/models


In [470]:
import joblib

for transaction_type, model in models_model2b.items():

    filename = (
        f"isolation_forest_{transaction_type.lower()}.joblib"
    )

    filepath = os.path.join(
        model_output_dir,
        filename
    )

    joblib.dump(model, filepath)

    print("Saved:", filename)

Saved: isolation_forest_cash_in.joblib
Saved: isolation_forest_cash_out.joblib
Saved: isolation_forest_debit.joblib
Saved: isolation_forest_payment.joblib
Saved: isolation_forest_transfer.joblib


In [471]:
import json

type_amount_stats = {}

for transaction_type in train_type_stats.index:

    type_amount_stats[transaction_type] = {
        "mean": float(
            train_type_stats.loc[
                transaction_type,
                "mean"
            ]
        ),
        "std": float(
            train_type_stats.loc[
                transaction_type,
                "std"
            ]
        )
    }


metadata = {
    "model_name": "Model 2B - Type-Specific Isolation Forests",

    "algorithm": "Isolation Forest",

    "threshold": -0.26,

    "features": numerical_features_model2b,

    "transaction_types": transaction_types,

    "type_amount_statistics": type_amount_stats,

    "training_period": {
        "start_step": 1,
        "end_step": 355
    },

    "validation_period": {
        "start_step": 356,
        "end_step": 549
    },

    "final_evaluation_period": {
        "start_step": 550,
        "end_step": 743
    }
}


metadata_path = os.path.join(
    model_output_dir,
    "model_metadata.json"
)

with open(metadata_path, "w") as file:
    json.dump(
        metadata,
        file,
        indent=4
    )

print("Metadata saved:", metadata_path)

Metadata saved: /kaggle/working/models/model_metadata.json


In [472]:
print(json.dumps(metadata, indent=4))

{
    "model_name": "Model 2B - Type-Specific Isolation Forests",
    "algorithm": "Isolation Forest",
    "threshold": -0.26,
    "features": [
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "orig_balance_change",
        "orig_balance_discrepancy",
        "amount_zscore_abs"
    ],
    "transaction_types": [
        "CASH_IN",
        "CASH_OUT",
        "DEBIT",
        "PAYMENT",
        "TRANSFER"
    ],
    "type_amount_statistics": {
        "CASH_IN": {
            "mean": 168764.15460343848,
            "std": 126598.7520730014
        },
        "CASH_OUT": {
            "mean": 176473.49823105746,
            "std": 156147.10668086464
        },
        "DEBIT": {
            "mean": 5592.754116435619,
            "std": 12626.19601060428
        },
        "PAYMENT": {
            "mean": 12854.768219393307,
            "std": 12390.728639387406
        },
        "TRANSFER": {
            "mean": 930792.0385174453,
            "std": 2023719.

In [473]:
loaded_models = {}

for transaction_type in transaction_types:

    filename = (
        f"isolation_forest_{transaction_type.lower()}.joblib"
    )

    filepath = os.path.join(
        model_output_dir,
        filename
    )

    loaded_models[transaction_type] = joblib.load(
        filepath
    )

print("Successfully loaded:", loaded_models.keys())

Successfully loaded: dict_keys(['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER'])


In [474]:
with open(metadata_path, "r") as file:
    loaded_metadata = json.load(file)

print("Threshold:", loaded_metadata["threshold"])
print("Features:", loaded_metadata["features"])

Threshold: -0.26
Features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'orig_balance_change', 'orig_balance_discrepancy', 'amount_zscore_abs']


In [475]:
for filename in os.listdir(model_output_dir):

    filepath = os.path.join(
        model_output_dir,
        filename
    )

    size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(
        f"{filename}: {size_mb:.2f} MB"
    )

isolation_forest_cash_out.joblib: 0.95 MB
isolation_forest_debit.joblib: 0.71 MB
model_metadata.json: 0.00 MB
isolation_forest_cash_in.joblib: 1.27 MB
isolation_forest_payment.joblib: 0.93 MB
isolation_forest_transfer.joblib: 0.65 MB


In [476]:
import shutil

shutil.make_archive(
    "/kaggle/working/models",
    "zip",
    "/kaggle/working/models"
)

print("Created /kaggle/working/models.zip")

Created /kaggle/working/models.zip
